In [ ]:
# On Google Colab, mount Drive and switch into the project directory.
# Locally this is a no-op (run the notebook from the `train/` directory).
try:
    from google.colab import drive
    import os
    drive.mount('/content/drive')
    os.chdir('/content/drive/MyDrive/Colab Notebooks')
except ModuleNotFoundError:
    pass

# Train LightGCN + CLIP multimodal hybrid (late fusion)

Plain LightGCN has graph-propagated collaborative-filtering signal but, like plain BPR, an
untrained cold item (< 5 train reviews) carries no useful information and scores near zero.

This notebook applies the same late-fusion recipe as `train-bpr-clip-hybrid.ipynb`, but the
collaborative score now comes from **LightGCN's graph-propagated embeddings** instead of raw
BPR dot products:

1. **CLIP projection init**: the layer-0 `item_embedding` (the ego embeddings propagated
   through the user-item graph) is initialized with a seeded random projection of the CLIP
   multimodal (text+image) embeddings, giving cold items a meaningful starting position.
2. **Per-user trainable alpha**: each user learns `alpha = sigmoid(user_alpha[u])` controlling
   how much to trust the LightGCN score vs the frozen CLIP content score.
3. **Z-scored rank fusion**: at full-sort eval the propagated-LightGCN scores and content
   scores are each z-scored per user before blending.

LightGCN's `reg_weight` L2 penalty on the ego embeddings is preserved (plain BPR has no such
term). The CLIP content branch is frozen — only the CF embeddings and alpha are trained.

In [ ]:
# ! pip install "pandas<=2.3.2" "numpy" "torch<=2.5" "scipy<1.12" "matplotlib" "seaborn" "matplotlib-venn" "datasets" "ipykernel" "recbole" "kmeans-pytorch" "sentence-transformers" "pyarrow"

In [1]:
import numpy as np

# For NumPy 2.0 compatibility with RecBole 1.2
np.float_ = np.float64
np.int_ = np.int64
np.complex_ = np.complex128
np.unicode_ = np.str_

# For SciPy 1.12+ compatibility: dok_matrix._update was removed
import scipy.sparse as sp
if not hasattr(sp.dok_matrix, '_update'):
    sp.dok_matrix._update = sp.dok_matrix.update

# Ensure logging on notebook works even on Colab
import logging
logging.getLogger().handlers.clear()

In [2]:
from typing import Any
import os
import torch
import torch.nn as nn
import pandas as pd
from recbole.config import Config
from recbole.data.dataloader import FullSortEvalDataLoader, AbstractDataLoader
from recbole.data import create_dataset, data_preparation
from recbole.model.general_recommender import LightGCN
from recbole.trainer import Trainer
from recbole.utils import init_seed, init_logger
from sentence_transformers import SentenceTransformer

/Users/yudhistiraonggowarsito/Documents/SMU/Courses/CS608 - Recommender Systems/grp_project/yc-code/amazon-item-recommender/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# --- Config ---
# Assume we have `*.train.inter`, `*.valid.inter`, `*.test.inter`
DATASET_NAME: str = "beauty"
DATA_DIR: str = "../data"
CLIP_TEXT_PATH: str = f"{DATA_DIR}/{DATASET_NAME}/clip_text_embeddings.pt"
SEED = 67
EMBEDDING_SIZE = 64  # trainable LightGCN embedding size
CLIP_DIM = 512

# if torch.cuda.is_available():
#     DEVICE = "cuda"
# elif torch.backends.mps.is_available():
#     DEVICE = "mps"
# else:
#     DEVICE = "cpu"

DEVICE = "cpu"

print(f"Using device: {DEVICE}")

Using device: cpu


## Create dataset

In [4]:
config_dict: dict[str, Any] = {
    "data_path": DATA_DIR,
    "dataset": DATASET_NAME,
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "benchmark_filename": ["train", "valid", "test"],
    "load_col": {
        "inter": ["user_id", "item_id"],
        "user": ["user_id", "cold"],
        "item": ["item_id", "title", "cold"],
    },
    "embedding_size": EMBEDDING_SIZE,
    "epochs": 10,
    "train_batch_size": 1024,
    "eval_batch_size": 409_600_000,
    "eval_args": {
        # Split is already determined by the `benchmark filename` as separate `.inter` files
        "split": None,
        "order": "TO",
        "mode": {"valid": "full", "test": "full"},
    },
    "metric_decimal_place": 6,
    "metrics": ["NDCG", "Recall", "MRR"],
    "topk": [20],
    "valid_metric": "NDCG@20",
    "n_layers": 3,
    "reg_weight": 1e-4,
    "seed": SEED,
}

# Borrow LightGCN's defaults (pairwise input + neg sampling); we subclass LightGCN below.
config: Config = Config(model="LightGCN", config_dict=config_dict)
config.final_config_dict["device"] = torch.device(DEVICE)

init_logger(config)
init_seed(SEED, reproducibility=True)

In [5]:
dataset = create_dataset(config)


def _normalize_cold_feature(feat: pd.DataFrame, field: str = "cold") -> None:
    """Recover the literal 0/1 warm/cold labels for `feat[field]`.

    The `cold` column is declared as a TOKEN field, so RecBole remaps the
    binary labels onto a shared token vocabulary (e.g.
    ``{'[PAD]': 0, '0': 1, '1': 2}``) and applies that remap inconsistently
    across the user/item feats: one keeps the raw '0'/'1' strings while the
    other ends up with remapped ids. Both forms break here — the raw strings
    can't be cast to a LongTensor in ``data_preparation``, and the remapped
    ids no longer match the literal 0.0/1.0 the warm/cold eval compares
    against. Map every value back to its original label via the vocabulary.
    """
    token_of_id = {i: t for t, i in dataset.field2token_id[field].items()}

    def to_label(v: object) -> int:
        token = v if isinstance(v, str) else token_of_id.get(int(v), str(v))
        return 0 if token == "[PAD]" else int(token)

    feat[field] = feat[field].map(to_label).astype("int64")


_normalize_cold_feature(dataset.user_feat)
_normalize_cold_feature(dataset.item_feat)

train_data, valid_data, test_data = data_preparation(config, dataset)

/Users/yudhistiraonggowarsito/Documents/SMU/Courses/CS608 - Recommender Systems/grp_project/yc-code/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/data/dataset/dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
23 Jun 23:22    INFO  [Training]: train_batch_size = [1024] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
23 Jun 23:22    INFO  [Evaluation]: eval_batch_size = [409600000] 

## CLIP-encode titles and fuse with precomputed image embeddings

Titles are encoded with CLIP's text tower directly. Image embeddings are loaded from
`clip_image_embeddings.pt` (built by the preprocessing notebook), re-indexed from
preprocessing iid to RecBole internal ID, then averaged with the text embeddings and
L2-normalized. Items without an image keep only their title embedding.

In [6]:
logging.getLogger('httpx').setLevel(logging.WARNING)

logger = logging.getLogger()

title_tokens: torch.Tensor = dataset.item_feat["title"]
id2token: dict[int, str] = {v: k for k, v in dataset.field2token_id["title"].items()}
titles: list[str] = [id2token.get(tok.item(), "") for tok in title_tokens]

if not os.path.exists(CLIP_TEXT_PATH):
    logger.info("Encoding %d item titles with CLIP text tower...", len(titles))
    clip_model = SentenceTransformer("clip-ViT-B-32", device=str(config["device"]))
    clip_text_embs = clip_model.encode(titles, show_progress_bar=True, batch_size=256, normalize_embeddings=True)
    clip_text_embs = torch.from_numpy(clip_text_embs).float()
    torch.save(clip_text_embs, CLIP_TEXT_PATH)
    logger.info("CLIP text embeddings saved to %s", CLIP_TEXT_PATH)
else:
    logger.info("CLIP text embeddings already exist at %s, loading cached.", CLIP_TEXT_PATH)

clip_text_embs = torch.load(CLIP_TEXT_PATH, map_location="cpu")

23 Jun 23:22    INFO  Encoding 250853 item titles with CLIP text tower...
23 Jun 23:22    INFO  Loading SentenceTransformer model from sentence-transformers/clip-ViT-B-32.
Batches: 100%|██████████| 980/980 [14:01<00:00,  1.16it/s]
23 Jun 23:36    INFO  CLIP text embeddings saved to ../data/beauty/clip_text_embeddings.pt
/var/folders/2v/617m0pq96qq942zr8nbfyzch0000gn/T/ipykernel_76789/3603690267.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitl

In [7]:
# --- Load CLIP image embeddings and fuse with text ---
clip_data = torch.load(f"{DATA_DIR}/beauty/clip_image_embeddings.pt", map_location="cpu")
clip_raw = clip_data["embeddings"]  # (n_dataset_items, 512)
iid_to_idx = clip_data["iid_to_idx"]

n_items = dataset.item_num
fused = clip_text_embs.clone()  # start from text
n_with_image = 0

for internal_id in range(1, n_items):
    token = dataset.id2token(dataset.iid_field, internal_id)
    try:
        iid = int(token)
    except ValueError:
        continue
    dense_idx = iid_to_idx.get(iid)
    if dense_idx is None:
        continue
    img_vec = clip_raw[dense_idx]
    if img_vec.abs().sum() == 0:
        continue
    fused[internal_id] = (clip_text_embs[internal_id] + img_vec) / 2.0
    n_with_image += 1

norms = fused.norm(dim=1, keepdim=True)
norms[norms == 0] = 1.0
fused = fused / norms
fused[0] = 0.0

print(f"Items with both text+image: {n_with_image:,} / {n_items - 1:,} ({100 * n_with_image / max(n_items - 1, 1):.2f}%)")
item_multimodal_embs = fused

/var/folders/2v/617m0pq96qq942zr8nbfyzch0000gn/T/ipykernel_76789/3801161301.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  clip_data = torch.load(f"{DATA_DIR}/beauty/cl

Items with both text+image: 29,321 / 250,852 (11.69%)


## Define LightGCNClipHybrid

Subclasses RecBole's `LightGCN`, reusing its graph propagation (`forward`), normalized
adjacency, `mf_loss` (BPR) and `reg_loss` (EmbLoss). Three things are added on top:

- The layer-0 `item_embedding` is warm-started with a seeded random projection of the CLIP
  multimodal embeddings (cold items start from a meaningful position in the graph input).
- A frozen per-user CLIP profile (mean of each user's *training* items' multimodal embeddings),
  computed once in `__init__`, plus a frozen item content table — the content signal that
  carries cold items.
- A trainable per-user `alpha = sigmoid(user_alpha[u])` blending the LightGCN score with the
  content score. LightGCN's `reg_weight` L2 penalty on the ego embeddings is preserved.

In [8]:
class LightGCNClipHybrid(LightGCN):
    def __init__(self, config, dataset, item_content_embeddings: torch.Tensor):
        # Parent builds user/item embeddings, norm_adj_matrix, losses, and runs
        # xavier init on the (as-yet content-free) parameters.
        super().__init__(config, dataset)

        content_dim = item_content_embeddings.shape[1]

        # Warm-start the layer-0 item ego embeddings with a seeded random projection
        # of the CLIP multimodal embeddings so cold items start from a meaningful spot.
        g = torch.Generator().manual_seed(config["seed"])
        projection = torch.randn(content_dim, self.latent_dim, generator=g) / (content_dim ** 0.5)
        item_init = item_content_embeddings.float() @ projection.to(item_content_embeddings.device)
        with torch.no_grad():
            self.item_embedding.weight.data.copy_(item_init)
        self.item_embedding.weight.data[0] = 0.0  # padding item stays inert in propagation

        # Per-user blend weight: sigmoid(user_alpha[u]) -> LightGCN vs CLIP content weight.
        self.user_alpha = nn.Embedding(self.n_users, 1)
        nn.init.constant_(self.user_alpha.weight, 0.0)

        # Frozen content signal: CLIP item embeddings + per-user mean profile over train items.
        self.register_buffer("item_content_embeddings", item_content_embeddings.float())

        train_inter = dataset.inter_feat
        users = train_inter[self.USER_ID]
        items = train_inter[self.ITEM_ID]
        profile_sum = torch.zeros(self.n_users, content_dim)
        profile_cnt = torch.zeros(self.n_users, 1)
        profile_sum.index_add_(0, users, self.item_content_embeddings[items])
        profile_cnt.index_add_(0, users, torch.ones_like(users, dtype=torch.float).unsqueeze(-1))
        self.register_buffer("user_content_profile", profile_sum / profile_cnt.clamp(min=1))

    @staticmethod
    def _row_normalize(x: torch.Tensor) -> torch.Tensor:
        """Z-score each row (each user's scores across candidate items)."""
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True)
        return (x - mean) / (std + 1e-8)

    def calculate_loss(self, interaction):
        # Clear the full-sort eval cache when training (mirrors parent LightGCN).
        if self.restore_user_e is not None or self.restore_item_e is not None:
            self.restore_user_e, self.restore_item_e = None, None

        user = interaction[self.USER_ID]
        pos_item = interaction[self.ITEM_ID]
        neg_item = interaction[self.NEG_ITEM_ID]

        # Graph-propagated collaborative score (LightGCN).
        user_all, item_all = self.forward()
        u_e = user_all[user]
        pos_e = item_all[pos_item]
        neg_e = item_all[neg_item]
        bpr_pos = torch.mul(u_e, pos_e).sum(dim=1)
        bpr_neg = torch.mul(u_e, neg_e).sum(dim=1)

        # Frozen CLIP content score.
        content_pos = (self.user_content_profile[user] * self.item_content_embeddings[pos_item]).sum(dim=-1)
        content_neg = (self.user_content_profile[user] * self.item_content_embeddings[neg_item]).sum(dim=-1)

        alpha = torch.sigmoid(self.user_alpha(user)).squeeze(-1)
        pos_score = alpha * bpr_pos + (1 - alpha) * content_pos
        neg_score = alpha * bpr_neg + (1 - alpha) * content_neg
        mf_loss = self.mf_loss(pos_score, neg_score)

        # L2 reg on the layer-0 ego embeddings, exactly like parent LightGCN.
        u_ego = self.user_embedding(user)
        pos_ego = self.item_embedding(pos_item)
        neg_ego = self.item_embedding(neg_item)
        reg_loss = self.reg_loss(u_ego, pos_ego, neg_ego, require_pow=self.require_pow)

        return mf_loss + self.reg_weight * reg_loss

    def predict(self, interaction):
        user = interaction[self.USER_ID]
        item = interaction[self.ITEM_ID]

        user_all, item_all = self.forward()
        bpr_score = torch.mul(user_all[user], item_all[item]).sum(dim=1)
        content_score = (self.user_content_profile[user] * self.item_content_embeddings[item]).sum(dim=-1)
        # Single-pair scores can't be row-normalized; blend raw (predict() is unused by the
        # full-sort eval this notebook relies on, kept only for API completeness).
        alpha = torch.sigmoid(self.user_alpha(user)).squeeze(-1)
        return alpha * bpr_score + (1 - alpha) * content_score

    def full_sort_predict(self, interaction):
        user = interaction[self.USER_ID]
        if self.restore_user_e is None or self.restore_item_e is None:
            self.restore_user_e, self.restore_item_e = self.forward()

        u_emb = self.restore_user_e[user]
        bpr_scores = torch.matmul(u_emb, self.restore_item_e.transpose(0, 1))
        content_scores = torch.matmul(self.user_content_profile[user], self.item_content_embeddings.t())

        alpha = torch.sigmoid(self.user_alpha(user))  # (B, 1)
        blended = alpha * self._row_normalize(bpr_scores) + (1 - alpha) * self._row_normalize(content_scores)
        return blended.view(-1)

## Train

In [9]:
model: LightGCNClipHybrid = LightGCNClipHybrid(
    config, train_data.dataset, item_multimodal_embs
).to(config["device"])
trainer: Trainer = Trainer(config, model)

best_valid_score, best_valid_result = trainer.fit(train_data, valid_data)

/Users/yudhistiraonggowarsito/Documents/SMU/Courses/CS608 - Recommender Systems/grp_project/yc-code/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/model/general_recommender/lightgcn.py:124: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:653.)
  SparseL = torch.sparse.FloatTensor(i, data, torch.Size(L.shape))
/Users/yudhistiraonggowarsito/Documents/SMU/Courses/CS608 - Recommender Systems/grp_project/yc-code/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/trainer/trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)
23 Jun 23:48    INFO  epoch 0 training [time: 716.09s, train loss: 474.2111]


In [10]:
print(f"\nBest valid score: {best_valid_score:.4f}")
print("Best valid result:")
for metric, score in best_valid_result.items():
    print(f"  {metric}: {score}")


Best valid score: 0.0086
Best valid result:
  ndcg@20: 0.008554
  recall@20: 0.016649
  mrr@20: 0.009642


## Evaluate on test set

In [11]:
test_result: dict[str, float] = trainer.evaluate(test_data, load_best_model=False)

print("Test results (Overall):")
for metric, value in test_result.items():
    print(f"  {metric}: {value}")

Test results (Overall):
  ndcg@20: 0.008047
  recall@20: 0.01469
  mrr@20: 0.009779


## Evaluate by warm/cold split

In [12]:
def evaluate_on_subset(
    data: AbstractDataLoader,
    mask: np.ndarray,
    label: str
):
    inter_feat = data.dataset.inter_feat
    cat_ds = data.dataset.copy(inter_feat[mask])
    cat_dl = FullSortEvalDataLoader(config, cat_ds, sampler=data._sampler)
    results = trainer.evaluate(cat_dl)
    rows.append({
        "Segment": label,
        "Interactions": int(mask.sum()),
        **results,
    })

rows = []

cold_to_label = {0.0: "warm", 1.0: "cold"}

def evaluate_by_column(entity_feat, id_field, inter_id_array, entity_name):
    id_to_cold = dict(zip(
        entity_feat[id_field].numpy(),
        entity_feat["cold"].numpy(),
    ))
    for cold_val, label in cold_to_label.items():
        ids = {eid for eid, c in id_to_cold.items() if c == cold_val}
        mask = np.isin(inter_id_array, list(ids))
        if not mask.any():
            print(f"  {entity_name}-{label}: no interactions — skipping")
            continue
        evaluate_on_subset(test_data, mask, f"{entity_name}-{label}")

# Evaluation by user segments
evaluate_by_column(
    dataset.user_feat, dataset.uid_field,
    test_data.dataset.inter_feat[dataset.uid_field].numpy(),
    "user"
)

# Evaluation by item segments
evaluate_by_column(
    dataset.item_feat, dataset.iid_field,
    test_data.dataset.inter_feat[dataset.iid_field].numpy(),
    "item"
)

# Cross-tabulation: user × item segments
uid_to_cold = dict(zip(
    dataset.user_feat[dataset.uid_field].numpy(),
    dataset.user_feat["cold"].numpy(),
))
iid_to_cold = dict(zip(
    dataset.item_feat[dataset.iid_field].numpy(),
    dataset.item_feat["cold"].numpy(),
))

uid_array = test_data.dataset.inter_feat[dataset.uid_field].numpy()
iid_array = test_data.dataset.inter_feat[dataset.iid_field].numpy()

for uc_val, uc_label in cold_to_label.items():
    for ic_val, ic_label in cold_to_label.items():
        uc_uids = {uid for uid, c in uid_to_cold.items() if c == uc_val}
        ic_iids = {iid for iid, c in iid_to_cold.items() if c == ic_val}
        mask = np.isin(uid_array, list(uc_uids)) & np.isin(iid_array, list(ic_iids))
        if not mask.any():
            print(f"  user-{uc_label}×item-{ic_label}: no interactions — skipping")
            continue
        evaluate_on_subset(test_data, mask, f"user-{uc_label}×item-{ic_label}")

# Display results sorted by NDCG
df_results = pd.DataFrame(rows).sort_values("ndcg@20", ascending=False)
display(df_results)

/Users/yudhistiraonggowarsito/Documents/SMU/Courses/CS608 - Recommender Systems/grp_project/yc-code/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/trainer/trainer.py:583: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue o

,Segment,Interactions,ndcg@20,recall@20,mrr@20
6,user-cold×item-warm,72341,0.012077,0.023353,0.012405
2,item-warm,88077,0.011992,0.023149,0.012748
4,user-warm×item-warm,15736,0.011418,0.021762,0.015068
1,user-cold,150964,0.008184,0.014985,0.009555
0,user-warm,46742,0.007139,0.012735,0.011256
7,user-cold×item-cold,78623,0.000097,0.000135,0.000140
3,item-cold,109629,0.000092,0.000146,0.000123
5,user-warm×item-cold,31006,0.000063,0.000208,0.000023
